In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv('data/Swan Consulting 1 - Project Data.csv')

In [4]:
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Count', 'State', 'Country', 'CustomerID'])
df_clean = df_clean.drop(columns= ['Lat Long', 'City','Zip Code'])
df_clean = df_clean.drop(columns= ['Churn Label'])
df_clean = df_clean.drop(columns= ['Churn Reason'])
df_clean = df_clean.drop(columns= ['Latitude', 'Longitude'])

In [5]:
df_clean['Gender'] = df_clean['Gender'].map({'Male':0, 'Female':1})
df_clean['Senior Citizen'] = df_clean['Senior Citizen'].map({'No':0, 'Yes':1})
df_clean['Partner'] = df_clean['Partner'].map({'No':0, 'Yes':1})
df_clean['Dependents'] = df_clean['Dependents'].map({'No':0, 'Yes':1})
df_clean['Phone Service'] = df_clean['Phone Service'].map({'No':0, 'Yes':1})
df_clean['Paperless Billing'] = df_clean['Paperless Billing'].map({'No':0, 'Yes':1})

In [6]:
df_clean['Multiple Lines'] = df_clean['Multiple Lines'].replace('No phone service', 'No')

service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                 'Tech Support', 'Streaming TV', 'Streaming Movies']
for col in service_cols:
    df_clean[col] = df_clean[col].replace('No internet service', 'No')

df_clean = pd.get_dummies(df_clean, columns=['Multiple Lines'], drop_first=True)
df_clean = pd.get_dummies(df_clean, columns=[
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies',
    'Contract',
    'Payment Method'
], drop_first=True)

In [7]:
df_clean = df_clean.astype({col: int for col in df_clean.select_dtypes(include='bool').columns})
df_clean[pd.to_numeric(df_clean['Total Charges'], errors='coerce').isna()][['Total Charges', 'Tenure Months']]
df_clean['Total Charges'] = pd.to_numeric(df_clean['Total Charges'], errors='coerce').fillna(0)

In [8]:
X = df_clean.drop(columns=['Churn Value'])
y = df_clean['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(X):
    X_const = X.assign(const=1)
    return pd.Series(
        [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1] - 1)],
        index=X.columns
    )

features = X_train.columns.tolist()
while True:
    vif = compute_vif(X_train[features])
    max_vif = vif.max()
    if max_vif <= 10:
        break
    drop_col = vif.idxmax()
    print(f"Dropping '{drop_col}' (VIF={max_vif:.1f})")
    features.remove(drop_col)

print("\nFinal VIF (all <= 10):")
print(vif.sort_values(ascending=False))

X_train = X_train[features]
X_test = X_test[features]
X = X[features]

Dropping 'Monthly Charges' (VIF=862.1)
Dropping 'Total Charges' (VIF=11.0)



Final VIF (all <= 10):
Tenure Months                             2.784120
Internet Service_No                       2.720147
Contract_Two year                         2.616028
Internet Service_Fiber optic              2.019145
Payment Method_Electronic check           1.970618
Payment Method_Mailed check               1.842719
Streaming Movies_Yes                      1.648983
Streaming TV_Yes                          1.635103
Contract_One year                         1.626575
Payment Method_Credit card (automatic)    1.554704
Tech Support_Yes                          1.496135
Device Protection_Yes                     1.494666
Multiple Lines_Yes                        1.430277
Online Security_Yes                       1.417632
Online Backup_Yes                         1.390898
Phone Service                             1.367833
Partner                                   1.338593
Dependents                                1.249669
Paperless Billing                         1.210664
Senior 

In [10]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

param_grid = {
    'scaler': [StandardScaler(), RobustScaler()],
    'model__C': [0.01, 0.1, 1, 10]
}

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)

{'model__C': 0.1, 'scaler': RobustScaler()}


In [12]:
scoring = ['accuracy', 'precision', 'recall', 'f1']
cv_results = cross_validate(grid_search.best_estimator_, X_train, y_train, cv=kfold, scoring=scoring)

for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.3f} +/- {scores.std():.3f}  {scores.round(3)}")

accuracy: 0.766 +/- 0.012  [0.777 0.749 0.752 0.776 0.774]
precision: 0.540 +/- 0.016  [0.555 0.519 0.523 0.552 0.553]
recall: 0.785 +/- 0.037  [0.816 0.732 0.776 0.836 0.766]
f1: 0.640 +/- 0.022  [0.66  0.607 0.624 0.665 0.642]


In [12]:
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1035
           1       0.51      0.76      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.79      0.74      0.76      1409



In [13]:
model = grid_search.best_estimator_.named_steps['model']
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coefficients)

                                   Feature  Coefficient
8             Internet Service_Fiber optic     0.797589
19         Payment Method_Electronic check     0.394268
6                        Paperless Billing     0.323466
15                    Streaming Movies_Yes     0.270213
14                        Streaming TV_Yes     0.255823
7                       Multiple Lines_Yes     0.255760
2                                  Partner     0.198394
1                           Senior Citizen     0.060703
20             Payment Method_Mailed check     0.043839
12                   Device Protection_Yes    -0.030180
0                                   Gender    -0.031626
18  Payment Method_Credit card (automatic)    -0.042617
11                       Online Backup_Yes    -0.105128
13                        Tech Support_Yes    -0.317952
5                            Phone Service    -0.344363
10                     Online Security_Yes    -0.352944
16                       Contract_One year    -0